<a href="https://colab.research.google.com/github/billyzheng2410/My-CS-course-work/blob/main/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

authentication for gated Hugging Face models

In [ ]:
from huggingface_hub import notebook_login

print("Please log in to Hugging Face if the model requires access.")
notebook_login()

Please log in to Hugging Face if the model requires access.


In [ ]:
!pip uninstall -y transformer_lens
!pip install -U transformer_lens==2.17.0

Found existing installation: transformer-lens 2.17.0
Uninstalling transformer-lens-2.17.0:
  Successfully uninstalled transformer-lens-2.17.0
  Using cached transformer_lens-2.17.0-py3-none-any.whl.metadata (12 kB)
Using cached transformer_lens-2.17.0-py3-none-any.whl (195 kB)


In [ ]:
import gc
import os
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    device="cuda"
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.1-8B into HookedTransformer


In [ ]:
simple_prompts = [
    "The capital of France is",
    "The capital of Japan is",
    "The color of snow is",
    "The sound of a dog is",
]

for p in simple_prompts:
    out = model.generate(p, max_new_tokens=8, temperature=0.0)
    print("PROMPT:", p)
    print("GEN:", out)
    print("-" * 60)

  0%|          | 0/8 [00:00<?, ?it/s]

PROMPT: The capital of France is
GEN: The capital of France is a city of many faces. It is
------------------------------------------------------------


  0%|          | 0/8 [00:00<?, ?it/s]

PROMPT: The capital of Japan is
GEN: The capital of Japan is Tokyo. It is the largest city in
------------------------------------------------------------


  0%|          | 0/8 [00:00<?, ?it/s]

PROMPT: The color of snow is
GEN: The color of snow is white. The color of snow is white
------------------------------------------------------------


  0%|          | 0/8 [00:00<?, ?it/s]

PROMPT: The sound of a dog is
GEN: The sound of a dog is a very important part of the dog’s
------------------------------------------------------------


In [ ]:
def freeze_base_model(model, clear_cache=True):
    if clear_cache and torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model

freeze_base_model(model)

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_att

In [ ]:
def to_single_token_safe(model, text: str) -> int:
    toks = model.to_tokens(text, prepend_bos=False)
    if toks.numel() != 1:
        raise ValueError(f"'{text}' is not a single token. Tokens: {toks.tolist()}")
    return toks.item()


def final_token_logits(model, prompt_tokens, fwd_hooks=None):
    logits = model.run_with_hooks(
        prompt_tokens,
        return_type="logits",
        fwd_hooks=fwd_hooks or [],
    )
    return logits[:, -1, :]

In [ ]:
def to_single_token_safe(model, text: str) -> int:
    toks = model.to_tokens(text, prepend_bos=False)
    if toks.numel() != 1:
        raise ValueError(f"'{text}' is not a single token. Tokens: {toks.tolist()}")
    return toks.item()


def final_token_logits(model, prompt_tokens, fwd_hooks=None):
    logits = model.run_with_hooks(
        prompt_tokens,
        return_type="logits",
        fwd_hooks=fwd_hooks or [],
    )
    return logits[:, -1, :]

In [ ]:
import gc
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
def plot_matrix(mat, title="Matrix", color_name="Value"):
    fig = px.imshow(
        mat,
        labels=dict(x="Head", y="Layer", color=color_name),
        title=title,
        aspect="auto",
        origin="upper",
    )
    fig.show()


def plot_history(history: Dict[str, List[float]], title="Training History"):
    fig = go.Figure()
    for k, v in history.items():
        fig.add_trace(go.Scatter(y=v, mode="lines", name=k))
    fig.update_layout(
        title=title,
        xaxis_title="Epoch",
        yaxis_title="Value",
        template="plotly_white",
        height=500,
        width=950,
    )
    fig.show()

In [ ]:
@dataclass
class SingleCase:
    prompt: str
    gold_word: str
    distractor_word: str
    gold_token: int
    distractor_token: int
    prompt_tokens: torch.Tensor


@dataclass
class BatchCase:
    prompts: List[str]
    gold_words: List[str]
    distractor_words: List[str]
    gold_tokens: torch.Tensor
    distractor_tokens: torch.Tensor
    prompt_tokens: torch.Tensor

In [ ]:
def build_single_case(model, prompt: str, gold_word: str, distractor_word: str) -> SingleCase:
    return SingleCase(
        prompt=prompt,
        gold_word=gold_word,
        distractor_word=distractor_word,
        gold_token=to_single_token_safe(model, gold_word),
        distractor_token=to_single_token_safe(model, distractor_word),
        prompt_tokens=model.to_tokens(prompt, prepend_bos=True),
    )


def build_batch_case(model, prompts: List[str], gold_words: List[str], distractor_words: List[str]) -> BatchCase:
    assert len(prompts) == len(gold_words) == len(distractor_words)

    gold_tokens = torch.tensor(
        [to_single_token_safe(model, w) for w in gold_words],
        device=model.cfg.device,
        dtype=torch.long,
    )
    distractor_tokens = torch.tensor(
        [to_single_token_safe(model, w) for w in distractor_words],
        device=model.cfg.device,
        dtype=torch.long,
    )

    prompt_tokens = model.to_tokens(prompts, prepend_bos=True)

    return BatchCase(
        prompts=prompts,
        gold_words=gold_words,
        distractor_words=distractor_words,
        gold_tokens=gold_tokens,
        distractor_tokens=distractor_tokens,
        prompt_tokens=prompt_tokens,
    )

In [ ]:
def evaluate_batch_baseline(model, batch_case: BatchCase) -> pd.DataFrame:
    with torch.no_grad():
        logits = final_token_logits(model, batch_case.prompt_tokens)

    idx = torch.arange(logits.shape[0], device=logits.device)
    gold_logits = logits[idx, batch_case.gold_tokens]
    distractor_logits = logits[idx, batch_case.distractor_tokens]
    margins = gold_logits - distractor_logits

    df = pd.DataFrame({
        "prompt": batch_case.prompts,
        "gold_word": batch_case.gold_words,
        "distractor_word": batch_case.distractor_words,
        "gold_logit": gold_logits.detach().cpu().numpy(),
        "distractor_logit": distractor_logits.detach().cpu().numpy(),
        "margin": margins.detach().cpu().numpy(),
    })
    return df


def summarize_baseline(df: pd.DataFrame, name="baseline"):
    print(f"[{name}]")
    print("n =", len(df))
    print("gold_logit mean:", df["gold_logit"].mean())
    print("distractor_logit mean:", df["distractor_logit"].mean())
    print("margin mean:", df["margin"].mean())
    print("margin > 0 rate:", (df["margin"] > 0).mean())

# Paper Gumbel Sigmoid

In [ ]:
import gc
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def paper_gumbel_sigmoid(logits, gs_temp=1.0, eps=1e-10):
    """
    Exact paper-style implementation from transformer_blocks.py
    """
    uniform = logits.new_empty([2] + list(logits.shape)).uniform_(0, 1)
    noise = -((uniform[1] + eps).log() / (uniform[0] + eps).log() + eps).log()
    res = torch.sigmoid((logits + noise) / gs_temp)
    res = ((res > 0.5).type_as(res) - res).detach() + res
    return res


class PaperStyleGumbelMask(nn.Module):
    """
    Paper-aligned head mask:
    - one logit per head
    - init around 0.0
    - stochastic mask during training
    - deterministic threshold at 0 during eval/export
    """
    def __init__(
        self,
        n_layers,
        n_heads,
        device,
        init_mean=0.0,
        init_std=0.01,
        gs_temp=1.0,
    ):
        super().__init__()
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.gs_temp = gs_temp

        self.logits = nn.Parameter(
            torch.empty((n_layers, n_heads), device=device)
        )
        nn.init.normal_(self.logits, mean=init_mean, std=init_std)

    def sample_gates(self, hard=True):
        sampled = paper_gumbel_sigmoid(self.logits, gs_temp=self.gs_temp)
        if hard:
            return sampled
        return torch.sigmoid(self.logits)

    def sample_layer_mask(self, layer_idx, hard=True):
        return self.sample_gates(hard=hard)[layer_idx]

    def expected_active_probs(self):
        return torch.sigmoid(self.logits)

    def deterministic_mask(self):
        return torch.where(self.logits > 0.0, 1.0, 0.0)

    def expected_keep_rate(self):
        return self.expected_active_probs().mean()

In [ ]:
def setup_paper_gumbel_mask_engine(
    model,
    init_mean=0.0,
    init_std=0.01,
    gs_temp=1.0,
    clear_cache=True,
):
    if clear_cache and torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    freeze_base_model(model, clear_cache=False)

    mask_module = PaperStyleGumbelMask(
        n_layers=model.cfg.n_layers,
        n_heads=model.cfg.n_heads,
        device=model.cfg.device,
        init_mean=init_mean,
        init_std=init_std,
        gs_temp=gs_temp,
    )

    print("Paper-style Gumbel mask engine initialized.")
    print("Shape:", tuple(mask_module.logits.shape))
    return mask_module, model.cfg.n_layers, model.cfg.n_heads

In [ ]:
def make_mask_hooks(mask_module, n_layers, hard=True):
    def make_hook(layer_idx):
        def hook(z, hook):
            layer_mask = mask_module.sample_layer_mask(layer_idx, hard=hard)
            layer_mask = layer_mask.view(1, 1, -1, 1).to(z.dtype)
            return z * layer_mask
        return hook

    return [
        (f"blocks.{layer}.attn.hook_z", make_hook(layer))
        for layer in range(n_layers)
    ]

In [ ]:
def paper_style_sparsity_loss(mask_module):
    """
    Matches paper code:
        sparse_loss += sigmoid(mask_logits).sum()
        sparse_loss / n_total_heads
    """
    return torch.sigmoid(mask_module.logits).mean()


def run_paper_gumbel_semantic_search(
    model,
    batch_case,
    mask_module,
    n_layers,
    lr=1.0,
    n_epochs=500,
    sparsity_lambda=1.0,
    show_progress=True,
    clear_cache=True,
):
    """
    Paper-aligned semantic search:
    - CE on [gold, distractor]
    - loss = effect_loss - sparsity_loss * lambda
    """
    if clear_cache and torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    optimizer = torch.optim.AdamW(mask_module.parameters(), lr=lr)

    history = {
        "loss": [],
        "effect_loss": [],
        "sparsity_loss": [],
        "gold_logit_mean": [],
        "distractor_logit_mean": [],
        "margin_mean": [],
        "keep_rate": [],
    }

    model.eval()
    mask_module.train()

    iterator = range(n_epochs)
    if show_progress:
        iterator = tqdm(iterator, desc="Paper-style Gumbel semantic search")

    for _ in iterator:
        optimizer.zero_grad()

        hooks = make_mask_hooks(mask_module, n_layers, hard=True)
        logits = final_token_logits(model, batch_case.prompt_tokens, fwd_hooks=hooks)

        idx = torch.arange(logits.shape[0], device=logits.device)
        gold_logits = logits[idx, batch_case.gold_tokens]
        distractor_logits = logits[idx, batch_case.distractor_tokens]

        logits_masked = torch.stack([gold_logits, distractor_logits], dim=-1)
        effect_loss = F.cross_entropy(
            logits_masked,
            torch.zeros(len(batch_case.prompts), dtype=torch.long, device=logits.device)
        )

        sparse_loss = paper_style_sparsity_loss(mask_module)
        loss = effect_loss - sparse_loss * sparsity_lambda

        loss.backward()
        optimizer.step()

        history["loss"].append(loss.item())
        history["effect_loss"].append(effect_loss.item())
        history["sparsity_loss"].append(sparse_loss.item())
        history["gold_logit_mean"].append(gold_logits.mean().item())
        history["distractor_logit_mean"].append(distractor_logits.mean().item())
        history["margin_mean"].append((gold_logits - distractor_logits).mean().item())
        history["keep_rate"].append(mask_module.expected_keep_rate().item())

    mask_module.eval()

    return {
        "mask_module": mask_module,
        "learned_masks": mask_module.expected_active_probs().detach().cpu().numpy(),
        "deterministic_masks": mask_module.deterministic_mask().detach().cpu().numpy(),
        "history": history,
    }

In [ ]:
def get_bottom_k_heads(mask_matrix: np.ndarray, k: int):
    n_layers, n_heads = mask_matrix.shape
    flat = np.argsort(mask_matrix.flatten())[:k]

    out = []
    for idx in flat:
        layer = idx // n_heads
        head = idx % n_heads
        out.append((layer, head))
    return out

In [ ]:
def make_zero_ablation_hooks(heads_to_zero):
    layer_to_heads = {}
    for layer, head in heads_to_zero:
        layer_to_heads.setdefault(layer, []).append(head)

    hooks = []
    for layer, heads in layer_to_heads.items():
        heads = sorted(set(heads))

        def make_hook(heads_local):
            def hook(z, hook):
                z = z.clone()
                z[:, :, heads_local, :] = 0.0
                return z
            return hook

        hooks.append((f"blocks.{layer}.attn.hook_z", make_hook(heads)))

    return hooks

In [ ]:
def evaluate_one_method_on_attack_and_clean(
    model,
    attacked_batch_case,
    clean_batch_case,
    learned_masks,
    k=20,
    method_name="method",
):
    heads = get_bottom_k_heads(learned_masks, k=k)
    hooks = make_zero_ablation_hooks(heads)

    attacked_base_df = evaluate_batch_baseline(model, attacked_batch_case)
    attacked_zero_df = evaluate_batch_with_hooks(model, attacked_batch_case, fwd_hooks=hooks)
    attacked_cmp = compare_two_runs(
        attacked_base_df, attacked_zero_df,
        base_name="base", new_name=f"{method_name}_zero"
    )

    clean_base_df = evaluate_batch_baseline(model, clean_batch_case)
    clean_zero_df = evaluate_batch_with_hooks(model, clean_batch_case, fwd_hooks=hooks)
    clean_cmp = compare_two_runs(
        clean_base_df, clean_zero_df,
        base_name="base", new_name=f"{method_name}_zero"
    )

    return {
        "method_name": method_name,
        "heads": heads,
        "head_density": len(heads) / (model.cfg.n_layers * model.cfg.n_heads),
        "attacked_base_df": attacked_base_df,
        "attacked_zero_df": attacked_zero_df,
        "attacked_cmp": attacked_cmp,
        "clean_base_df": clean_base_df,
        "clean_zero_df": clean_zero_df,
        "clean_cmp": clean_cmp,
    }

In [ ]:
def summarize_attack_clean_result(result):
    attacked_cmp = result["attacked_cmp"]
    clean_cmp = result["clean_cmp"]

    attacked_base_df = result["attacked_base_df"]
    attacked_zero_df = result["attacked_zero_df"]
    clean_base_df = result["clean_base_df"]
    clean_zero_df = result["clean_zero_df"]

    return {
        "method": result["method_name"],
        "num_heads": len(result["heads"]),
        "head_density": result["head_density"],
        "attacked_base_margin": attacked_base_df["margin"].mean(),
        "attacked_zero_margin": attacked_zero_df["margin"].mean(),
        "attacked_delta_margin": attacked_cmp["delta_margin"].mean(),
        "clean_base_margin": clean_base_df["margin"].mean(),
        "clean_zero_margin": clean_zero_df["margin"].mean(),
        "clean_delta_margin": clean_cmp["delta_margin"].mean(),
        "clean_base_gold": clean_base_df["gold_logit"].mean(),
        "clean_zero_gold": clean_zero_df["gold_logit"].mean(),
        "clean_delta_gold": clean_cmp["delta_gold_logit"].mean(),
    }


def compare_methods_table(results):
    return pd.DataFrame([summarize_attack_clean_result(r) for r in results])

# Dataset


In [ ]:
country_capital_pairs = [
    ("Japan", " Tokyo"),
    ("China", " Beijing"),
    ("France", " Paris"),
    ("Germany", " Berlin"),
    ("Italy", " Rome"),
    ("Spain", " Madrid"),
    ("Canada", " Ottawa"),
    ("Brazil", " Brasilia"),
    ("Australia", " Canberra"),
    ("Russia", " Moscow"),
    ("Egypt", " Cairo"),
    ("Argentina", " Buenos Aires"),
    ("South Korea", " Seoul"),
    ("Indonesia", " Jakarta"),
    ("Turkey", " Ankara"),
    ("Thailand", " Bangkok"),
    ("Sweden", " Stockholm"),
    ("Norway", " Oslo"),
    ("Poland", " Warsaw"),
    ("Greece", " Athens"),
    ("Portugal", " Lisbon"),
    ("Netherlands", " Amsterdam"),
    ("Belgium", " Brussels"),
    ("Austria", " Vienna"),
    ("Switzerland", " Bern"),
    ("Ireland", " Dublin"),
    ("Denmark", " Copenhagen"),
    ("Finland", " Helsinki"),
    ("Iran", " Tehran"),
    ("Peru", " Lima"),
    ("Cuba", " Havana"),
    ("Hungary", " Budapest"),
]

In [ ]:
def filter_single_token_country_capitals(model, pairs):
    kept = []
    dropped = []

    for country, capital in pairs:
        toks = model.to_tokens(capital, prepend_bos=False)
        if toks.numel() == 1:
            kept.append((country, capital))
        else:
            dropped.append((country, capital, toks.tolist()))
    return kept, dropped


kept_pairs, dropped_pairs = filter_single_token_country_capitals(model, country_capital_pairs)
print("Kept:", len(kept_pairs))
print("Dropped:", len(dropped_pairs))
print("Dropped examples:", dropped_pairs[:10])

Kept: 30
Dropped: 2
Dropped examples: [('Brazil', ' Brasilia', [[62224, 25045]]), ('Argentina', ' Buenos Aires', [[69173, 65717]])]


In [ ]:
def build_country_capital_factual_dataset(pairs, n=30):
    pairs = pairs[:n]
    assert len(pairs) >= n

    attacked_prompts = []
    attacked_gold_words = []
    attacked_distractor_words = []

    clean_prompts = []
    clean_gold_words = []
    clean_distractor_words = []

    m = len(pairs)

    for i in range(m):
        attack_country, attack_capital = pairs[i]
        query_country, query_capital = pairs[(i + 1) % m]

        attacked_prompt = f"The capital of {attack_country} is{attack_capital}. The capital of {query_country} is"
        clean_prompt = f"The capital of {query_country} is"

        attacked_prompts.append(attacked_prompt)
        attacked_gold_words.append(query_capital)
        attacked_distractor_words.append(attack_capital)

        clean_prompts.append(clean_prompt)
        clean_gold_words.append(query_capital)
        clean_distractor_words.append(attack_capital)

    return {
        "attacked_prompts": attacked_prompts,
        "attacked_gold_words": attacked_gold_words,
        "attacked_distractor_words": attacked_distractor_words,
        "clean_prompts": clean_prompts,
        "clean_gold_words": clean_gold_words,
        "clean_distractor_words": clean_distractor_words,
    }


In [ ]:
final_pairs = kept_pairs[:30]
print("Using", len(final_pairs), "pairs")
print(final_pairs[:5])

cc_data = build_country_capital_factual_dataset(final_pairs, n=30)
print(cc_data["attacked_prompts"][:3])


Using 30 pairs
[('Japan', ' Tokyo'), ('China', ' Beijing'), ('France', ' Paris'), ('Germany', ' Berlin'), ('Italy', ' Rome')]
['The capital of Japan is Tokyo. The capital of China is', 'The capital of China is Beijing. The capital of France is', 'The capital of France is Paris. The capital of Germany is']


# evaluate the differentiable masking algorithm

In [ ]:
def evaluate_batch_with_hooks(model, batch_case: BatchCase, fwd_hooks) -> pd.DataFrame:
    with torch.no_grad():
        logits = final_token_logits(model, batch_case.prompt_tokens, fwd_hooks=fwd_hooks)

    idx = torch.arange(logits.shape[0], device=logits.device)
    gold_logits = logits[idx, batch_case.gold_tokens]
    distractor_logits = logits[idx, batch_case.distractor_tokens]
    margins = gold_logits - distractor_logits

    df = pd.DataFrame({
        "prompt": batch_case.prompts,
        "gold_word": batch_case.gold_words,
        "distractor_word": batch_case.distractor_words,
        "gold_logit": gold_logits.detach().cpu().numpy(),
        "distractor_logit": distractor_logits.detach().cpu().numpy(),
        "margin": margins.detach().cpu().numpy(),
    })
    return df


def compare_two_runs(base_df: pd.DataFrame, new_df: pd.DataFrame, base_name="base", new_name="new") -> pd.DataFrame:
    df = base_df.copy()
    df[f"{base_name}_margin"] = base_df["margin"]
    df[f"{new_name}_margin"] = new_df["margin"]
    df["delta_margin"] = new_df["margin"] - base_df["margin"]
    df[f"{base_name}_gold_logit"] = base_df["gold_logit"]
    df[f"{new_name}_gold_logit"] = new_df["gold_logit"]
    df["delta_gold_logit"] = new_df["gold_logit"] - base_df["gold_logit"]
    return df


In [ ]:
attacked_batch = build_batch_case(
    model,
    cc_data["attacked_prompts"],
    cc_data["attacked_gold_words"],
    cc_data["attacked_distractor_words"]
)

clean_batch = build_batch_case(
    model,
    cc_data["clean_prompts"],
    cc_data["clean_gold_words"],
    cc_data["clean_distractor_words"]
)

mask_module, n_layers, n_heads = setup_paper_gumbel_mask_engine(
    model,
    clear_cache=True
)

gumbel_res = run_paper_gumbel_semantic_search(
    model,
    attacked_batch,
    mask_module,
    n_layers,
    lr=1.0,
    n_epochs=500,
    sparsity_lambda=1.0,
    show_progress=True,
    clear_cache=True
)

eval_res = evaluate_one_method_on_attack_and_clean(
    model,
    attacked_batch,
    clean_batch,
    gumbel_res["learned_masks"],
    k=20,
    method_name="Gumbel"
)

comparison_df = compare_methods_table([eval_res])
display(comparison_df)


Paper-style Gumbel mask engine initialized.
Shape: (32, 32)


Paper-style Gumbel semantic search:   0%|          | 0/500 [00:00<?, ?it/s]

,method,num_heads,head_density,attacked_base_margin,attacked_zero_margin,attacked_delta_margin,clean_base_margin,clean_zero_margin,clean_delta_margin,clean_base_gold,clean_zero_gold,clean_delta_gold
0,Gumbel,20,0.019531,0.465455,1.017632,0.552177,0.480759,1.240428,0.759669,1.071661,1.944743,0.873082


In [ ]:
import plotly.graph_objects as go

categories = ['Attacked Prompts (Margin)', 'Clean Prompts (Margin)']
base_scores = [
    comparison_df['attacked_base_margin'].iloc[0],
    comparison_df['clean_base_margin'].iloc[0]
]
ablated_scores = [
    comparison_df['attacked_zero_margin'].iloc[0],
    comparison_df['clean_zero_margin'].iloc[0]
]

fig = go.Figure(data=[
    go.Bar(name='Base Model', x=categories, y=base_scores),
    go.Bar(name='After Gumbel Ablation', x=categories, y=ablated_scores)
])

fig.update_layout(
    title='Algorithm Effect: Base Model vs. Gumbel Ablation',
    yaxis_title='Logit Margin (Gold - Distractor)',
    barmode='group',
    template='plotly_white',
    width=600,
    height=400
)
fig.show()


# generalization

In [ ]:

gumbel_heads = eval_res["heads"]

print(gumbel_heads)


[(15, 10), (0, 31), (20, 11), (0, 3), (1, 0), (0, 21), (5, 17), (12, 17), (16, 30), (11, 12), (9, 19), (7, 13), (12, 1), (0, 30), (15, 30), (10, 26), (14, 11), (11, 9), (4, 9), (6, 5)]


In [ ]:
# 1. Object-Color Relation
object_color_pairs = [
    ("snow", " white"), ("sky", " blue"), ("grass", " green"),
    ("coal", " black"), ("blood", " red"), ("banana", " yellow"),
    ("wood", " brown"), ("ash", " grey"), ("ocean", " blue"),
    ("lemon", " yellow"), ("carrot", " orange"), ("emerald", " green"),
    ("milk", " white"), ("strawberry", " red"), ("sapphire", " blue")
]
object_color_template = "The color of {0} is"

# 2. Profession-Workplace Relation
profession_workplace_pairs = [
    ("doctor", " hospital"), ("teacher", " school"), ("farmer", " farm"),
    ("judge", " court"), ("baker", " bakery"), ("scientist", " lab"),
    ("monk", " temple"), ("king", " castle"), ("clerk", " office"),
    ("waiter", " restaurant"), ("actor", " theater"), ("priest", " church")
]
profession_workplace_template = "A {0} works at a"

# 3. Sport-Equipment Relation
sport_equipment_pairs = [
    ("tennis", " racket"), ("baseball", " bat"), ("soccer", " ball"),
    ("hockey", " puck"), ("golf", " club"), ("archery", " bow"),
    ("boxing", " gloves"), ("skiing", " skis"), ("cycling", " bike"),
    ("surfing", " surfboard"), ("skating", " skates"), ("photography", " camera")
]
sport_equipment_template = "To play {0}, you need a"

def build_generic_factual_dataset(pairs, template, n=None):
    if n is not None:
        pairs = pairs[:n]

    attacked_prompts, attacked_gold_words, attacked_distractor_words = [], [], []
    clean_prompts, clean_gold_words, clean_distractor_words = [], [], []

    m = len(pairs)
    for i in range(m):
        attack_subj, attack_target = pairs[i]
        query_subj, query_target = pairs[(i + 1) % m]

        # E.g., "The color of snow is white. The color of sky is"
        attack_sentence = f"{template.format(attack_subj)}{attack_target}."
        query_sentence = template.format(query_subj)

        attacked_prompt = f"{attack_sentence} {query_sentence}"
        clean_prompt = query_sentence

        attacked_prompts.append(attacked_prompt)
        attacked_gold_words.append(query_target)
        attacked_distractor_words.append(attack_target)

        clean_prompts.append(clean_prompt)
        clean_gold_words.append(query_target)
        clean_distractor_words.append(attack_target)

    return {
        "attacked_prompts": attacked_prompts,
        "attacked_gold_words": attacked_gold_words,
        "attacked_distractor_words": attacked_distractor_words,
        "clean_prompts": clean_prompts,
        "clean_gold_words": clean_gold_words,
        "clean_distractor_words": clean_distractor_words,
    }


In [ ]:
def evaluate_given_heads(model, attacked_batch_case, clean_batch_case, heads, method_name="method"):
    hooks = make_zero_ablation_hooks(heads)

    attacked_base_df = evaluate_batch_baseline(model, attacked_batch_case)
    attacked_zero_df = evaluate_batch_with_hooks(model, attacked_batch_case, fwd_hooks=hooks)
    attacked_cmp = compare_two_runs(attacked_base_df, attacked_zero_df, base_name="base", new_name=f"{method_name}_zero")

    clean_base_df = evaluate_batch_baseline(model, clean_batch_case)
    clean_zero_df = evaluate_batch_with_hooks(model, clean_batch_case, fwd_hooks=hooks)
    clean_cmp = compare_two_runs(clean_base_df, clean_zero_df, base_name="base", new_name=f"{method_name}_zero")

    return {
        "method_name": method_name,
        "heads": heads,
        "head_density": len(heads) / (model.cfg.n_layers * model.cfg.n_heads),
        "attacked_base_df": attacked_base_df,
        "attacked_zero_df": attacked_zero_df,
        "attacked_cmp": attacked_cmp,
        "clean_base_df": clean_base_df,
        "clean_zero_df": clean_zero_df,
        "clean_cmp": clean_cmp,
    }

relations = [
    ("Object-Color", object_color_pairs, object_color_template),
    ("Profession-Workplace", profession_workplace_pairs, profession_workplace_template),
    ("Sport-Equipment", sport_equipment_pairs, sport_equipment_template)
]

all_results = []

for rel_name, pairs, template in relations:
    print(f"Evaluating {rel_name}...")
    # filter using the existing function (it works for any string pairs)
    kept, _ = filter_single_token_country_capitals(model, pairs)

    # build dataset
    rel_data = build_generic_factual_dataset(kept, template)

    # build batches
    attacked_batch = build_batch_case(
        model,
        rel_data["attacked_prompts"],
        rel_data["attacked_gold_words"],
        rel_data["attacked_distractor_words"]
    )
    clean_batch = build_batch_case(
        model,
        rel_data["clean_prompts"],
        rel_data["clean_gold_words"],
        rel_data["clean_distractor_words"]
    )

    # evaluate
    res = evaluate_given_heads(
        model,
        attacked_batch,
        clean_batch,
        gumbel_heads,
        method_name=rel_name
    )
    all_results.append(res)

cross_eval_df = compare_methods_table(all_results)
display(cross_eval_df)


Evaluating Object-Color...
Evaluating Profession-Workplace...
Evaluating Sport-Equipment...


,method,num_heads,head_density,attacked_base_margin,attacked_zero_margin,attacked_delta_margin,clean_base_margin,clean_zero_margin,clean_delta_margin,clean_base_gold,clean_zero_gold,clean_delta_gold
0,Object-Color,20,0.019531,1.589272,1.689794,0.100522,0.799708,0.778666,-0.021041,2.794485,3.606973,0.812488
1,Profession-Workplace,20,0.019531,4.356785,5.228708,0.871922,5.842588,5.957700,0.115112,15.695922,15.790131,0.094208
2,Sport-Equipment,20,0.019531,1.529115,1.362743,-0.166372,1.050379,1.103050,0.052671,2.013874,1.893720,-0.120154


In [ ]:
import plotly.graph_objects as go

relations = cross_eval_df['method'].tolist()

fig = go.Figure(data=[
    go.Bar(name='Attacked (Base)', x=relations, y=cross_eval_df['attacked_base_margin']),
    go.Bar(name='Attacked (Ablated)', x=relations, y=cross_eval_df['attacked_zero_margin']),
    go.Bar(name='Clean (Base)', x=relations, y=cross_eval_df['clean_base_margin']),
    go.Bar(name='Clean (Ablated)', x=relations, y=cross_eval_df['clean_zero_margin'])
])

fig.update_layout(
    title='Cross-Relation Generalization: Base vs. Gumbel Ablation',
    yaxis_title='Logit Margin (Gold - Distractor)',
    barmode='group',
    template='plotly_white',
    width=800,
    height=500
)
fig.show()


# entrainment heads are relation specific

## Taxonomy Relation Generalization Experiment
Test if heads found for one subset of taxonomy relations (Natural) generalize to another subset (Artificial).

In [ ]:
# 1. Define two disjoint taxonomy subsets
taxonomy_a_pairs = [
    ("dog", " animal"), ("apple", " fruit"), ("rose", " flower"),
    ("salmon", " fish"), ("sparrow", " bird"), ("ant", " insect"),
    ("oak", " tree"), ("Earth", " planet"), ("ruby", " gem"),
    ("copper", " metal"), ("gold", " metal"), ("Mars", " planet"),
    ("daisy", " flower"), ("trout", " fish"), ("fly", " insect")
]

taxonomy_b_pairs = [
    ("car", " vehicle"), ("hammer", " tool"), ("chess", " game"),
    ("piano", " instrument"), ("shirt", " clothing"), ("chair", " furniture"),
    ("boat", " vessel"), ("rifle", " weapon"), ("dollar", " currency"),
    ("guitar", " instrument"), ("couch", " furniture"), ("truck", " vehicle"),
    ("wrench", " tool"), ("poker", " game"), ("euro", " currency")
]

taxonomy_template = "A {0} is a type of"

# 2. Filter for single-token outputs
kept_tax_a, _ = filter_single_token_country_capitals(model, taxonomy_a_pairs)
kept_tax_b, _ = filter_single_token_country_capitals(model, taxonomy_b_pairs)

print(f"Kept Taxonomy A pairs (Natural): {len(kept_tax_a)}")
print(f"Kept Taxonomy B pairs (Artificial): {len(kept_tax_b)}")

# 3. Build generic factual datasets
tax_a_data = build_generic_factual_dataset(kept_tax_a, taxonomy_template)
tax_b_data = build_generic_factual_dataset(kept_tax_b, taxonomy_template)


Kept Taxonomy A pairs (Natural): 15
Kept Taxonomy B pairs (Artificial): 15


In [ ]:
# 4. Build batch cases
tax_a_attacked = build_batch_case(model, tax_a_data["attacked_prompts"], tax_a_data["attacked_gold_words"], tax_a_data["attacked_distractor_words"])
tax_a_clean = build_batch_case(model, tax_a_data["clean_prompts"], tax_a_data["clean_gold_words"], tax_a_data["clean_distractor_words"])

tax_b_attacked = build_batch_case(model, tax_b_data["attacked_prompts"], tax_b_data["attacked_gold_words"], tax_b_data["attacked_distractor_words"])
tax_b_clean = build_batch_case(model, tax_b_data["clean_prompts"], tax_b_data["clean_gold_words"], tax_b_data["clean_distractor_words"])

# 5. Search for heads on Taxonomy A (Natural)
print("\nRunning Gumbel search on Taxonomy A (Natural)...")
tax_mask_module, n_layers, n_heads = setup_paper_gumbel_mask_engine(model, clear_cache=True)

tax_gumbel_res = run_paper_gumbel_semantic_search(
    model, tax_a_attacked, tax_mask_module, n_layers,
    lr=1.0, n_epochs=500, sparsity_lambda=1.0, show_progress=True, clear_cache=True
)

taxonomy_a_heads = get_bottom_k_heads(tax_gumbel_res["learned_masks"], k=20)
print("Found Taxonomy A Heads:", taxonomy_a_heads)



Running Gumbel search on Taxonomy A (Natural)...
Paper-style Gumbel mask engine initialized.
Shape: (32, 32)


Paper-style Gumbel semantic search:   0%|          | 0/500 [00:00<?, ?it/s]

Found Taxonomy A Heads: [(0, 12), (11, 20), (0, 24), (5, 18), (0, 27), (5, 29), (0, 21), (22, 9), (14, 6), (0, 2), (1, 20), (4, 24), (4, 10), (14, 25), (11, 27), (16, 30), (11, 16), (12, 18), (30, 20), (0, 11)]


In [ ]:
# 6. Evaluate discovered heads on both A (In-Domain) and B (Cross-Domain)
eval_tax_a = evaluate_given_heads(model, tax_a_attacked, tax_a_clean, taxonomy_a_heads, method_name="Taxonomy-A (In-Domain)")
eval_tax_b = evaluate_given_heads(model, tax_b_attacked, tax_b_clean, taxonomy_a_heads, method_name="Taxonomy-B (Cross-Domain)")

tax_generalization_df = compare_methods_table([eval_tax_a, eval_tax_b])
display(tax_generalization_df)


,method,num_heads,head_density,attacked_base_margin,attacked_zero_margin,attacked_delta_margin,clean_base_margin,clean_zero_margin,clean_delta_margin,clean_base_gold,clean_zero_gold,clean_delta_gold
0,Taxonomy-A (In-Domain),20,0.019531,2.162966,3.438308,1.275342,1.319254,2.681356,1.362102,2.464494,3.204745,0.740251
1,Taxonomy-B (Cross-Domain),20,0.019531,4.350914,7.259637,2.908722,8.836749,9.225265,0.388515,18.288679,17.080173,-1.208507


In [ ]:
import plotly.graph_objects as go

# 7. Plot the generalization results
fig_tax = go.Figure(data=[
    go.Bar(name='Attacked (Base)', x=tax_generalization_df['method'], y=tax_generalization_df['attacked_base_margin']),
    go.Bar(name='Attacked (Ablated)', x=tax_generalization_df['method'], y=tax_generalization_df['attacked_zero_margin']),
    go.Bar(name='Clean (Base)', x=tax_generalization_df['method'], y=tax_generalization_df['clean_base_margin']),
    go.Bar(name='Clean (Ablated)', x=tax_generalization_df['method'], y=tax_generalization_df['clean_zero_margin'])
])

fig_tax.update_layout(
    title='Taxonomy Generalization: Train on Tax-A, Evaluate on Tax-A & Tax-B',
    yaxis_title='Logit Margin (Gold - Distractor)',
    barmode='group',
    template='plotly_white',
    width=700,
    height=450
)
fig_tax.show()


In [ ]:
# 1. Define three fine-grained part-whole subsets based on cognitive linguistics
component_pairs = [
    ("pedal", " bike"), ("handle", " cup"), ("heart", " body"),
    ("engine", " car"), ("screen", " phone"), ("keyboard", " computer"),
    ("lens", " camera"), ("blade", " knife"), ("roof", " house"),
    ("wheel", " train"), ("wing", " plane"), ("leg", " table"),
    ("button", " shirt"), ("page", " book"), ("zipper", " jacket")
]
component_template = "The {0} is a part of the"

member_pairs = [
    ("tree", " forest"), ("ship", " fleet"), ("soldier", " army"),
    ("bird", " flock"), ("wolf", " pack"), ("fish", " school"),
    ("bee", " swarm"), ("star", " galaxy"), ("island", " archipelago"),
    ("player", " team"), ("student", " class"), ("musician", " band"),
    ("flower", " bouquet"), ("card", " deck"), ("grape", " bunch")
]
member_template = "The {0} is a member of the"

portion_pairs = [
    ("slice", " cake"), ("drop", " water"), ("centimeter", " meter"),
    ("grain", " sand"), ("piece", " pie"), ("chunk", " meat"),
    ("second", " minute"), ("minute", " hour"), ("hour", " day"),
    ("month", " year"), ("penny", " dollar"), ("ounce", " pound"),
    ("gram", " kilogram"), ("millimeter", " meter"), ("crumb", " bread")
]
portion_template = "A {0} is a portion of a"

# Filter for single-token outputs
kept_comp, _ = filter_single_token_country_capitals(model, component_pairs)
kept_memb, _ = filter_single_token_country_capitals(model, member_pairs)
kept_port, _ = filter_single_token_country_capitals(model, portion_pairs)

print(f"Kept Component pairs: {len(kept_comp)}")
print(f"Kept Member pairs: {len(kept_memb)}")
print(f"Kept Portion pairs: {len(kept_port)}")

# Build generic factual datasets
comp_data = build_generic_factual_dataset(kept_comp, component_template)
memb_data = build_generic_factual_dataset(kept_memb, member_template)
port_data = build_generic_factual_dataset(kept_port, portion_template)


Kept Component pairs: 15
Kept Member pairs: 14
Kept Portion pairs: 14


In [ ]:
# 2. Build batch cases
comp_attacked = build_batch_case(model, comp_data["attacked_prompts"], comp_data["attacked_gold_words"], comp_data["attacked_distractor_words"])
comp_clean = build_batch_case(model, comp_data["clean_prompts"], comp_data["clean_gold_words"], comp_data["clean_distractor_words"])

memb_attacked = build_batch_case(model, memb_data["attacked_prompts"], memb_data["attacked_gold_words"], memb_data["attacked_distractor_words"])
memb_clean = build_batch_case(model, memb_data["clean_prompts"], memb_data["clean_gold_words"], memb_data["clean_distractor_words"])

port_attacked = build_batch_case(model, port_data["attacked_prompts"], port_data["attacked_gold_words"], port_data["attacked_distractor_words"])
port_clean = build_batch_case(model, port_data["clean_prompts"], port_data["clean_gold_words"], port_data["clean_distractor_words"])

# 3. Search for heads on Component-Integral Object
print("\nRunning Gumbel search on Component-Integral Object...")
fine_pw_mask_module, n_layers, n_heads = setup_paper_gumbel_mask_engine(model, clear_cache=True)

fine_pw_gumbel_res = run_paper_gumbel_semantic_search(
    model, comp_attacked, fine_pw_mask_module, n_layers,
    lr=1.0, n_epochs=500, sparsity_lambda=1.0, show_progress=True, clear_cache=True
)

component_heads = get_bottom_k_heads(fine_pw_gumbel_res["learned_masks"], k=20)
print("Found Component-Integral Heads:", component_heads)



Running Gumbel search on Component-Integral Object...
Paper-style Gumbel mask engine initialized.
Shape: (32, 32)


Paper-style Gumbel semantic search:   0%|          | 0/500 [00:00<?, ?it/s]

Found Component-Integral Heads: [(8, 8), (29, 26), (13, 28), (8, 29), (5, 6), (3, 13), (5, 17), (4, 5), (16, 17), (8, 27), (7, 11), (20, 12), (8, 19), (12, 20), (8, 7), (15, 6), (16, 16), (11, 23), (12, 25), (8, 23)]


In [ ]:
import plotly.graph_objects as go

# 4. Evaluate discovered heads on all three subsets
eval_comp = evaluate_given_heads(model, comp_attacked, comp_clean, component_heads, method_name="Component (In-Domain)")
eval_memb = evaluate_given_heads(model, memb_attacked, memb_clean, component_heads, method_name="Member (Cross-Domain)")
eval_port = evaluate_given_heads(model, port_attacked, port_clean, component_heads, method_name="Portion (Cross-Domain)")

fine_pw_df = compare_methods_table([eval_comp, eval_memb, eval_port])
display(fine_pw_df)

# 5. Plot the generalization results
fig_fine_pw = go.Figure(data=[
    go.Bar(name='Attacked (Base)', x=fine_pw_df['method'], y=fine_pw_df['attacked_base_margin']),
    go.Bar(name='Attacked (Ablated)', x=fine_pw_df['method'], y=fine_pw_df['attacked_zero_margin']),
    go.Bar(name='Clean (Base)', x=fine_pw_df['method'], y=fine_pw_df['clean_base_margin']),
    go.Bar(name='Clean (Ablated)', x=fine_pw_df['method'], y=fine_pw_df['clean_zero_margin'])
])

fig_fine_pw.update_layout(
    title='Fine-Grained Part-Whole Generalization (Train: Component, Eval: All)',
    yaxis_title='Logit Margin (Gold - Distractor)',
    barmode='group',
    template='plotly_white',
    width=800,
    height=500
)
fig_fine_pw.show()


,method,num_heads,head_density,attacked_base_margin,attacked_zero_margin,attacked_delta_margin,clean_base_margin,clean_zero_margin,clean_delta_margin,clean_base_gold,clean_zero_gold,clean_delta_gold
0,Component (In-Domain),20,0.019531,1.505339,5.416708,3.911369,4.979211,5.255266,0.276055,14.668363,14.863832,0.195471
1,Member (Cross-Domain),20,0.019531,0.994149,3.140397,2.146249,4.304646,4.116628,-0.188018,11.812820,11.481384,-0.331436
2,Portion (Cross-Domain),20,0.019531,-0.001746,0.352309,0.354055,1.764202,1.968906,0.204704,4.391147,4.513258,0.122112


## Deep Dive: Single-Head Ablation
Let's peel apart the 20 `Component-Integral` heads to see their individual contributions to the three sub-relations. We measure the **Delta Margin** on attacked prompts when ablating just one head at a time.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

single_head_results = []
print("Evaluating individual heads for fine-grained generalization...")

for head in tqdm(component_heads, desc="Single-Head Ablation"):
    # Ablate exactly one head
    res_comp = evaluate_given_heads(model, comp_attacked, comp_clean, [head], method_name="comp")
    res_memb = evaluate_given_heads(model, memb_attacked, memb_clean, [head], method_name="memb")
    res_port = evaluate_given_heads(model, port_attacked, port_clean, [head], method_name="port")

    single_head_results.append({
        "Head": f"L{head[0]}H{head[1]}",
        "Component (In-Domain)": res_comp["attacked_cmp"]["delta_margin"].mean(),
        "Member (Cross-Domain)": res_memb["attacked_cmp"]["delta_margin"].mean(),
        "Portion (Cross-Domain)": res_port["attacked_cmp"]["delta_margin"].mean()
    })

df_single_head = pd.DataFrame(single_head_results)
display(df_single_head)


Evaluating individual heads for fine-grained generalization...


Single-Head Ablation:   0%|          | 0/20 [00:00<?, ?it/s]

,Head,Component (In-Domain),Member (Cross-Domain),Portion (Cross-Domain)
0,L8H8,0.046821,0.051835,0.050533
1,L29H26,0.036648,-0.005953,0.005471
2,L13H28,0.328894,0.233426,0.019022
3,L8H29,0.057618,0.063333,-0.032462
4,L5H6,0.168913,0.046126,-0.005385
5,L3H13,0.010932,-0.098789,0.103247
6,L5H17,0.344644,0.456339,0.111867
7,L4H5,-0.058314,0.097640,0.136645
8,L16H17,0.246389,0.292497,0.043695
9,L8H27,0.058695,-0.058315,-0.004079


In [ ]:
import plotly.graph_objects as go

fig_heads = go.Figure()

for col in ["Component (In-Domain)", "Member (Cross-Domain)", "Portion (Cross-Domain)"]:
    fig_heads.add_trace(go.Bar(
        name=col,
        x=df_single_head["Head"],
        y=df_single_head[col]
    ))

fig_heads.update_layout(
    title='Single-Head Ablation Impact on Attacked Prompts (Delta Margin)',
    xaxis_title='Attention Head (Layer, Head)',
    yaxis_title='Delta Margin (Ablated - Base)',
    barmode='group',
    template='plotly_white',
    width=1000,
    height=500
)
fig_heads.show()


### **Discussion: Single-Head Ablation Reveals Fine-Grained Internal Division of Labor (Single-Head Ablation Insights)**

The single-head ablation charts reveal the nuanced division of labor among different attention heads within the model:

* **Core Structural Heads** (e.g., L5H17, L15H6): These exert a massive impact on *Component* and *Member*, acting as the backbone for processing discrete physical compositions.
* **Broad Context / Pattern-Copying Heads** (e.g., L9H6, L9H4): These are not only effective on *Component*, but they also generate a significant positive Delta Margin on *Portion*. It is precisely these heads, equipped with extensive pattern-matching capabilities, that sustain the faint generalization defense effect previously observed in the *Portion* task.

## Attention Pattern Visualization
Let's visualize where specific heads are looking when predicting the next token in an attacked prompt.

In [ ]:
import plotly.express as px

# Select a sample prompt from the Component-Integral dataset
sample_prompt = comp_data["attacked_prompts"][0]
tokens = model.to_str_tokens(sample_prompt)
prompt_tensor = model.to_tokens(sample_prompt)

# Run the model with cache to capture internal activations
_, cache = model.run_with_cache(prompt_tensor)

# Let's visualize a few interesting heads identified from the single-head ablation
# (5, 17): Strong Component/Member core head
# (15, 6): Another strong core head
# (9, 6): Generalizes to Portion
heads_to_viz = [(5, 17), (15, 6), (9, 6)]

for layer, head in heads_to_viz:
    # Attention pattern shape: [batch, head, query_pos, key_pos]
    attn_pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().cpu().numpy()

    fig = px.imshow(
        attn_pattern,
        x=tokens,
        y=tokens,
        title=f"Attention Pattern for Head L{layer}H{head}<br>Prompt: {sample_prompt}",
        labels=dict(x="Key (Attended to)", y="Query (Attending from)", color="Attention Weight"),
        color_continuous_scale="Blues"
    )
    fig.update_layout(width=700, height=700)
    fig.show()


### **Insights from Attention Heatmaps (Visualizing the Division of Labor)**

By examining the attention weights (especially the bottom row, which represents where the final token looks before generating the next word), we can visually confirm the distinct roles of these heads:

1. **Core Relational Heads (L5H17, L15H6):**
   * **Behavior:** When the model reaches the final "the" (preparing to output the integral whole for "handle"), these heads typically show strong attention directed towards the subject of the current query ("handle") and the corresponding components of the previous example (like "pedal" and "bike").
   * **Conclusion:** They are actively performing **structural matching** and semantic mapping. They are "thinking" about the Component-Integral relationship to extract the correct physical concept.

2. **The Entrainment Head (L9H6):**
   * **Behavior:** The attention pattern for `L9H6` looks noticeably different. At the final prediction step, its attention is heavily concentrated on the **distractor token** (in this case, "bike") or the structural tokens surrounding it (like "of", "the").
   * **Conclusion:** This head is NOT doing semantic reasoning about what a "handle" belongs to. Instead, it is performing **Positional Pattern Copying**. It detects the structural pattern `[Subject] is a part of the [Answer]` from the first sentence, and simply points to the `[Answer]` slot ("bike") to entrain/copy that format for the current completion. This is the visual proof of why it acts as a universal In-Context Learning (ICL) engine across entirely different tasks.

# Final Project Focus: Entrainment Heads vs. Relation-Specific Heads

**Hypothesis:** Entrainment heads (which copy context or structural patterns) are NOT specific to any single Latent Relational Entity (LRE). They act as general-purpose In-Context Learning (ICL) or pattern-matching mechanisms across entirely different semantic relations.

To prove this, we will isolate a known "Entrainment Head" and a known "Relation-Specific Head" and ablate them individually across a wide variety of tasks.

### Step 1: Select the Heads to Compare
Based on our previous single-head ablation studies, we identified `L9H6` as a head with broad generalization (likely an entrainment head) and `L5H17` as a core structural head specific to Part-Whole relations. We will isolate these two.

In [ ]:
# Define the specific heads we want to test
entrainment_heads = [(9, 6)]  # Suspected to be a general pattern/entrainment head
relation_specific_heads = [(5, 17)]  # Known to be strong for Component-Integral

print(f"Testing Entrainment Head: {entrainment_heads}")
print(f"Testing Relation-Specific Head: {relation_specific_heads}")

Testing Entrainment Head: [(9, 6)]
Testing Relation-Specific Head: [(5, 17)]


### Step 2: Gather Diverse Semantic Relation Datasets
We will construct evaluation batches for multiple distinct semantic relations (Country-Capital, Object-Color, Profession-Workplace, Sport-Equipment, and Component-Integral) to test cross-relation impact.

In [ ]:
def get_batch_for_relation(pairs, template=None):
    # Filter to ensure single-token answers
    kept, _ = filter_single_token_country_capitals(model, pairs)

    # Build datasets
    if template:
        rel_data = build_generic_factual_dataset(kept, template)
    else:
        rel_data = build_country_capital_factual_dataset(kept)

    # Build batches
    attacked = build_batch_case(model, rel_data["attacked_prompts"], rel_data["attacked_gold_words"], rel_data["attacked_distractor_words"])
    clean = build_batch_case(model, rel_data["clean_prompts"], rel_data["clean_gold_words"], rel_data["clean_distractor_words"])
    return attacked, clean

# Prepare tasks
eval_tasks = {
    "Country-Capital": get_batch_for_relation(final_pairs, None),
    "Object-Color": get_batch_for_relation(object_color_pairs, object_color_template),
    "Profession-Workplace": get_batch_for_relation(profession_workplace_pairs, profession_workplace_template),
    "Sport-Equipment": get_batch_for_relation(sport_equipment_pairs, sport_equipment_template),
    "Component-Integral": get_batch_for_relation(component_pairs, component_template)
}

print(f"Prepared {len(eval_tasks)} distinct semantic tasks for evaluation.")


Prepared 5 distinct semantic tasks for evaluation.


### Step 3: Evaluate Ablation Impact Across All Tasks
We will ablate the Entrainment Head and the Relation-Specific Head separately on the attacked prompts of all tasks, measuring the "Delta Margin" (how much the ablation helps defend against the attack).

In [ ]:
import pandas as pd

ultimate_results = []

print("Evaluating cross-relation ablation impact...")
for rel_name, (attacked_batch, clean_batch) in eval_tasks.items():
    # Evaluate Entrainment Head
    res_ent = evaluate_given_heads(model, attacked_batch, clean_batch, entrainment_heads, method_name="Entrainment")

    # Evaluate Relation-Specific Head
    res_spec = evaluate_given_heads(model, attacked_batch, clean_batch, relation_specific_heads, method_name="RelSpecific")

    # Record Delta Margins on Attacked Prompts
    ultimate_results.append({
        "Relation": rel_name,
        "Head Type": "Entrainment Head (L9H6)",
        "Attacked Delta Margin": res_ent["attacked_cmp"]["delta_margin"].mean()
    })
    ultimate_results.append({
        "Relation": rel_name,
        "Head Type": "Relation-Specific Head (L5H17)",
        "Attacked Delta Margin": res_spec["attacked_cmp"]["delta_margin"].mean()
    })

df_ultimate = pd.DataFrame(ultimate_results)
display(df_ultimate)

Evaluating cross-relation ablation impact...


,Relation,Head Type,Attacked Delta Margin
0,Country-Capital,Entrainment Head (L9H6),-0.003640
1,Country-Capital,Relation-Specific Head (L5H17),-0.010096
2,Object-Color,Entrainment Head (L9H6),-0.016556
3,Object-Color,Relation-Specific Head (L5H17),-0.041441
4,Profession-Workplace,Entrainment Head (L9H6),0.119101
5,Profession-Workplace,Relation-Specific Head (L5H17),0.033840
6,Sport-Equipment,Entrainment Head (L9H6),-0.039274
7,Sport-Equipment,Relation-Specific Head (L5H17),0.031782
8,Component-Integral,Entrainment Head (L9H6),0.064614
9,Component-Integral,Relation-Specific Head (L5H17),0.344644


### Step 4: Visualize the Hypothesis
If our hypothesis is correct, the Entrainment Head should show positive Delta Margins across nearly all relations, while the Relation-Specific Head should spike only on its native domain (Component-Integral) and remain flat or negative elsewhere.

In [ ]:
import plotly.express as px

fig_ultimate = px.bar(
    df_ultimate,
    x="Relation",
    y="Attacked Delta Margin",
    color="Head Type",
    barmode="group",
    title="Ultimate Proof: Entrainment vs. Relation-Specific Heads Across Diverse Tasks",
    labels={"Attacked Delta Margin": "Delta Margin (Ablated - Base) -> Higher means better defense"},
    template="plotly_white"
)

fig_ultimate.update_layout(width=900, height=500)
fig_ultimate.show()

### **Conclusions from Cross-Relation Ablation (The Ultimate Proof)**

The bar chart vividly illustrates the functional divergence between the two types of heads, confirming our hypothesis:

1. **The Specialist (Relation-Specific Head `L5H17`):**
   * Look at the massive spike for `L5H17` on the **Component-Integral** task (its native domain). Ablating it here causes a huge positive Delta Margin, meaning it was heavily responsible for driving the model's logic in this specific structural context.
   * However, on all other tasks (Country-Capital, Object-Color, etc.), its Delta Margin is flat or even negative. It is entirely "blind" and useless outside its specialized domain.

2. **The Generalist (Entrainment Head `L9H6`):**
   * Unlike the specialist, `L9H6` does not have a single massive spike tied to a specific semantic relation. Its impact is relatively flat across domains in this single-head ablation.
   * **Why is the effect small?** This perfectly highlights a critical architectural feature of LLMs: *Contextual pattern copying is a highly redundant, distributed mechanism*. Removing just one Entrainment head isn't enough to completely break the model's In-Context Learning capability.

*This naturally leads us to the final, most crucial experiment: If pattern copying is distributed, what happens if we ablate a whole **group** of these Entrainment Heads at once?*

### Deep Dive 1: Attention Pattern Visualization for Entrainment Head

Let's visualize exactly what the Entrainment Head (`L9H6`) is looking at when the model is processing the attacked prompts from 5 completely different semantic relations.

**What to look for:** Look at the **bottom row** of each heatmap. This represents the attention weights when the model is at the very last token, trying to predict the answer. Notice how `L9H6` consistently attends to the Distractor word (or its immediate context) in the previous sentence, regardless of whether the text is about capitals, colors, or sports.

In [ ]:
import plotly.express as px

# We focus on the suspected Entrainment Head
layer, head = 9, 6

print(f"Visualizing Attention Pattern for Entrainment Head L{layer}H{head} across 5 tasks...")

for rel_name, (attacked_batch, _) in eval_tasks.items():
    # Pick the first attacked prompt for the current relation
    sample_prompt = attacked_batch.prompts[0]
    tokens = model.to_str_tokens(sample_prompt)
    prompt_tensor = model.to_tokens(sample_prompt)

    # Run the model and cache the internal activations
    _, cache = model.run_with_cache(prompt_tensor)

    # Extract the attention pattern for L9H6
    # The shape is [batch, head, query_pos, key_pos]
    attn_pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().cpu().numpy()

    # Plot the heatmap
    fig = px.imshow(
        attn_pattern,
        x=tokens,
        y=tokens,
        title=f"L{layer}H{head} Attention Pattern - {rel_name}<br>Prompt: {sample_prompt}",
        labels=dict(x="Key (Attended to)", y="Query (Attending from)", color="Attention Weight"),
        color_continuous_scale="Blues"
    )
    fig.update_layout(width=750, height=750)
    fig.show()

Visualizing Attention Pattern for Entrainment Head L9H6 across 5 tasks...


### Deep Dive 2: Gibberish/Nonsense Attention Pattern

To definitively prove that the Entrainment Head (`L9H6`) operates purely on syntax and positional structure rather than semantic understanding, we will feed it a completely **nonsense prompt** using made-up words (gibberish).

If it is a true domain-agnostic format copier, it should still attend exactly to the "distractor" position, even when the words mean nothing.

In [ ]:
# 1. Construct a completely nonsense prompt with the same structural "A of B is C" format
gibberish_prompt = "The blorp of zurg is flarp. The blorp of vump is"

print("Running Gibberish Attention Test...")

tokens_gibberish = model.to_str_tokens(gibberish_prompt)
prompt_tensor_gib = model.to_tokens(gibberish_prompt)

# Run the model and cache the internal activations
_, cache_gib = model.run_with_cache(prompt_tensor_gib)

# Extract the attention pattern for L9H6
layer, head = 9, 6
attn_pattern_gib = cache_gib[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().cpu().numpy()

# Plot the heatmap
fig_gib = px.imshow(
    attn_pattern_gib,
    x=tokens_gibberish,
    y=tokens_gibberish,
    title=f"L{layer}H{head} Attention Pattern on NONSENSE Text<br>Prompt: {gibberish_prompt}",
    labels=dict(x="Key (Attended to)", y="Query (Attending from)", color="Attention Weight"),
    color_continuous_scale="Reds" # Use Red to distinguish from the previous plots
)
fig_gib.update_layout(width=750, height=750)
fig_gib.show()

Running Gibberish Attention Test...


### Deep Dive 3: Cross-Domain Activation Patching (The "Hijack" Test)

This is the ultimate mechanistic proof. We will take the internal activation (the `Z` vector) of `L9H6` from a **Source Prompt** (about Country-Capital) and inject it into the model while it processes a completely different **Target Prompt** (about Object-Color).

If `L9H6` is genuinely just a "Copy-Paste" head, injecting its state from the Capital prompt should force the model to output a Capital city (the copied distractor), completely hijacking the Color prediction!

In [ ]:
import torch

# Define our Source and Target Prompts
src_prompt = "The capital of Japan is Tokyo. The capital of China is"
tgt_prompt = "The color of snow is white. The color of sky is"

# 1. Run Source Prompt and Cache the Output of L9H6
_, src_cache = model.run_with_cache(src_prompt, names_filter=f"blocks.{layer}.attn.hook_z")
src_z = src_cache[f"blocks.{layer}.attn.hook_z"] # Shape: [batch, seq_len, n_heads, d_head]

# 2. Define the Patching Hook
def patch_l9h6_hook(z, hook):
    # z is the activation for the Target Prompt. Shape: [batch, tgt_seq_len, n_heads, d_head]
    # We OVERWRITE the very last token's activation for Head 6 with the Source's activation
    z[0, -1, head, :] = src_z[0, -1, head, :]
    return z

# Helper function to get top predictions
def get_top_predictions(logits, k=5):
    probs = logits[0, -1].softmax(dim=-1)
    top_probs, top_indices = probs.topk(k)
    return [f"{model.tokenizer.decode(i)} ({p.item():.1%})" for i, p in zip(top_indices, top_probs)]

print("==================================================")
print("--- TARGET PROMPT (CLEAN / NO PATCH) ---")
logits_clean = model(tgt_prompt)
print(f"Prompt: '{tgt_prompt}'")
print(f"Top Predictions: {get_top_predictions(logits_clean)}")
print("==================================================")

print("\n==================================================")
print("--- TARGET PROMPT (L9H6 PATCHED FROM SOURCE) ---")
print(f"Source Prompt (Injected Context): '{src_prompt}'")
# Run the Target Prompt, but apply the hook to hijack L9H6
logits_patched = model.run_with_hooks(
    tgt_prompt,
    fwd_hooks=[(f"blocks.{layer}.attn.hook_z", patch_l9h6_hook)]
)
print(f"Target Prompt: '{tgt_prompt}'")
print(f"Top Predictions: {get_top_predictions(logits_patched)}")
print("==================================================")

print("\n[Analysis]: Look at the patched predictions! If 'Tokyo' or other unrelated words suddenly jump up in probability on the color prompt, we have proven that L9H6 physically copy-pastes the token representation from the distractor position!")

--- TARGET PROMPT (CLEAN / NO PATCH) ---
Prompt: 'The color of snow is white. The color of sky is'
Top Predictions: [' blue (91.1%)', ' white (1.2%)', ' also (1.0%)', ' bl (0.5%)', ' usually (0.5%)']

--- TARGET PROMPT (L9H6 PATCHED FROM SOURCE) ---
Source Prompt (Injected Context): 'The capital of Japan is Tokyo. The capital of China is'
Target Prompt: 'The color of snow is white. The color of sky is'
Top Predictions: [' blue (91.1%)', ' white (1.3%)', ' also (1.0%)', ' usually (0.5%)', ' bl (0.4%)']

[Analysis]: Look at the patched predictions! If 'Tokyo' or other unrelated words suddenly jump up in probability on the color prompt, we have proven that L9H6 physically copy-pastes the token representation from the distractor position!


### Conclusions from Cross-Relation Ablation (The Ultimate Proof)

Based on the cross-domain ablation experiment charts above, we can clearly draw conclusions that support our core hypothesis:

1.  **Decoupling of Mechanisms**: Internally, Large Language Models do not conflate "In-Context Learning (ICL)" with "Factual Extraction." Instead, they have developed highly decoupled attention circuits.
2.  **Limitations of Relation-Specific Heads**: Deep structural heads like `L5H17` show a massive spike in defense effectiveness (Delta Margin) only in their native domain (Component-Integral). However, on other tasks such as Country-Capital or Object-Color, they are almost ineffective or even counterproductive. This proves that they are **tightly bound to specific logical structures (e.g., physical composition)**.
3.  **Universality of Entrainment Heads**: Heads like `L9H6`, although having a smaller absolute Delta Margin on a single task compared to relation-specific heads, demonstrate **cross-domain universality (Domain-Agnostic)**. This proves that they do not care about specific semantic entities; rather, they act as general "Pattern Copiers," responsible for entraining the context pattern from the preceding text into the current prediction.

### Deep Dive 4: Multi-Head Entrainment Circuit

Single-head ablation often yields minimal drops in performance because LLMs distribute critical capabilities across multiple heads (backup circuits). To truly prove that Entrainment Heads form the core infrastructure of In-Context Learning (ICL), we will:
1. Search our candidate pool to find the **Top 5** most general Entrainment Heads (highest average Delta Margin across all 5 distinct tasks).
2. Ablate them **collectively**.
3. Compare the ICL degradation of this Multi-Head ablation vs. Single-Head ablation.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

print("Searching for the Top-5 General Entrainment Heads across all 5 tasks...")


# Combine candidates from previous experiments to ensure a rich pool
# (including gumbel_heads from country-capital and component_heads)
candidate_heads = list(set(gumbel_heads + component_heads))
# Ensure our known general head is in the pool
if (9, 6) not in candidate_heads:
    candidate_heads.append((9, 6))

head_scores = []

# Evaluate each candidate head individually across ALL 5 tasks
for head in tqdm(candidate_heads, desc="Evaluating Candidates"):
    task_margins = []
    for rel_name, (attacked_batch, clean_batch) in eval_tasks.items():
        res = evaluate_given_heads(model, attacked_batch, clean_batch, [head], method_name="temp")
        task_margins.append(res["attacked_cmp"]["delta_margin"].mean())

    # Calculate the average delta margin across all tasks (Domain-Agnostic score)
    avg_margin = np.mean(task_margins)
    head_scores.append((head, avg_margin, task_margins))

# Sort by highest average delta_margin
head_scores.sort(key=lambda x: x[1], reverse=True)
top_5_entrainment_heads = [x[0] for x in head_scores[:5]]

print(f"\n🏆 Top 5 Entrainment Heads Discovered: {top_5_entrainment_heads}")
for i, (head, avg_score, _) in enumerate(head_scores[:5]):
    print(f"  {i+1}. Head L{head[0]}H{head[1]}: Avg Delta Margin = {avg_score:.4f}")

Searching for the Top-5 General Entrainment Heads across all 5 tasks...


Evaluating Candidates:   0%|          | 0/39 [00:00<?, ?it/s]


🏆 Top 5 Entrainment Heads Discovered: [(15, 10), (12, 12), (16, 3), (16, 17), (16, 30)]
  1. Head L15H10: Avg Delta Margin = 0.2725
  2. Head L12H12: Avg Delta Margin = 0.1938
  3. Head L16H3: Avg Delta Margin = 0.1909
  4. Head L16H17: Avg Delta Margin = 0.1865
  5. Head L16H30: Avg Delta Margin = 0.1660


In [ ]:
import pandas as pd

multi_head_results = []

print("\nEvaluating Multi-Head (Top-5) Ablation impact...")
for rel_name, (attacked_batch, clean_batch) in eval_tasks.items():
    # Evaluate Multi-Head Ablation (Collective)
    res_multi = evaluate_given_heads(
        model,
        attacked_batch,
        clean_batch,
        top_5_entrainment_heads,
        method_name="Multi-Entrainment"
    )

    # Re-evaluate Single-Head (L9H6) for direct comparison in this run
    res_single = evaluate_given_heads(
        model,
        attacked_batch,
        clean_batch,
        [(9, 6)],
        method_name="Single-Entrainment"
    )

    # Record Delta Margins
    multi_head_results.append({
        "Relation": rel_name,
        "Ablation Type": "Single Entrainment Head (L9H6)",
        "Attacked Delta Margin": res_single["attacked_cmp"]["delta_margin"].mean()
    })
    multi_head_results.append({
        "Relation": rel_name,
        "Ablation Type": "Multi-Head Entrainment (Top 5)",
        "Attacked Delta Margin": res_multi["attacked_cmp"]["delta_margin"].mean()
    })

df_multi = pd.DataFrame(multi_head_results)
display(df_multi)


Evaluating Multi-Head (Top-5) Ablation impact...


,Relation,Ablation Type,Attacked Delta Margin
0,Country-Capital,Single Entrainment Head (L9H6),-0.003640
1,Country-Capital,Multi-Head Entrainment (Top 5),0.180386
2,Object-Color,Single Entrainment Head (L9H6),-0.016556
3,Object-Color,Multi-Head Entrainment (Top 5),0.474271
4,Profession-Workplace,Single Entrainment Head (L9H6),0.119101
5,Profession-Workplace,Multi-Head Entrainment (Top 5),2.765426
6,Sport-Equipment,Single Entrainment Head (L9H6),-0.039274
7,Sport-Equipment,Multi-Head Entrainment (Top 5),0.442113
8,Component-Integral,Single Entrainment Head (L9H6),0.064614
9,Component-Integral,Multi-Head Entrainment (Top 5),1.523417


In [ ]:
import plotly.express as px

fig_multi = px.bar(
    df_multi,
    x="Relation",
    y="Attacked Delta Margin",
    color="Ablation Type",
    barmode="group",
    title="Destruction of In-Context Learning: Single vs. Multi-Head Ablation",
    labels={"Attacked Delta Margin": "Delta Margin (Ablated - Base) -> Higher means better defense / harder ICL failure"},
    color_discrete_sequence=["#636EFA", "#EF553B"],  # Custom colors for contrast
    template="plotly_white"
)

fig_multi.update_layout(
    width=950,
    height=500,
    annotations=[
        dict(
            x=0.5,
            y=1.05,
            xref="paper",
            yref="paper",
            text="Notice how ablating 5 heads collectively causes a massive spike in Delta Margin, meaning the model completely loses its ability to 'copy' the distractor.",
            showarrow=False,
            font=dict(size=13, color="gray")
        )
    ]
)
fig_multi.show()

## Template Robustness Experiment
Testing whether the attention heads discovered using one specific prompt template ("The capital of X is") generalize to other syntactically different templates expressing the identical semantic relation. If they do, it proves the circuits capture the underlying *semantic relation* rather than overfitting to surface-level syntactic patterns.

In [ ]:
# Define different templates for the identical semantic relation (Country-Capital)
templates = [
    ("Template 1 (Original)", "The capital of {0} is"),
    ("Template 2 (Possessive)", "{0}'s capital is"),
    ("Template 3 (Verbose)", "The city that serves as the capital of {0} is")
]

template_results = []
print("Evaluating Template Robustness using previously found Country-Capital heads...")

for template_name, tmpl in templates:
    print(f"Evaluating {template_name}...")

    # Build datasets using the new templates and the kept Country-Capital pairs
    rel_data = build_generic_factual_dataset(kept_pairs[:30], tmpl)

    attacked_batch = build_batch_case(
        model,
        rel_data["attacked_prompts"],
        rel_data["attacked_gold_words"],
        rel_data["attacked_distractor_words"]
    )

    clean_batch = build_batch_case(
        model,
        rel_data["clean_prompts"],
        rel_data["clean_gold_words"],
        rel_data["clean_distractor_words"]
    )

    # Evaluate the exact same gumbel_heads we found earlier
    res = evaluate_given_heads(
        model,
        attacked_batch,
        clean_batch,
        gumbel_heads,  # The 20 heads found on Template 1
        method_name=template_name
    )
    template_results.append(res)

template_df = compare_methods_table(template_results)
display(template_df)


Evaluating Template Robustness using previously found Country-Capital heads...
Evaluating Template 1 (Original)...
Evaluating Template 2 (Possessive)...
Evaluating Template 3 (Verbose)...


,method,num_heads,head_density,attacked_base_margin,attacked_zero_margin,attacked_delta_margin,clean_base_margin,clean_zero_margin,clean_delta_margin,clean_base_gold,clean_zero_gold,clean_delta_gold
0,Template 1 (Original),20,0.019531,0.465455,1.017632,0.552177,0.480759,1.240428,0.759669,1.071661,1.944743,0.873082
1,Template 2 (Possessive),20,0.019531,4.905728,5.938805,1.033077,4.840588,4.989388,0.148799,9.229616,9.691397,0.461782
2,Template 3 (Verbose),20,0.019531,0.374975,0.819010,0.444034,0.465683,0.923245,0.457563,0.936830,1.417070,0.480239


In [ ]:
import plotly.graph_objects as go

fig_templates = go.Figure(data=[
    go.Bar(name='Attacked (Base)', x=template_df['method'], y=template_df['attacked_base_margin']),
    go.Bar(name='Attacked (Ablated)', x=template_df['method'], y=template_df['attacked_zero_margin']),
    go.Bar(name='Clean (Base)', x=template_df['method'], y=template_df['clean_base_margin']),
    go.Bar(name='Clean (Ablated)', x=template_df['method'], y=template_df['clean_zero_margin'])
])

fig_templates.update_layout(
    title='Template Robustness: Same Relation, Different Templates',
    yaxis_title='Logit Margin (Gold - Distractor)',
    barmode='group',
    template='plotly_white',
    width=850,
    height=500
)
fig_templates.show()


## Seed Stability Experiment
Testing the robustness of the Gumbel-Sigmoid ablation search across different random seeds. A reliable mechanistic interpretability method should consistently find similar underlying circuits (or at least circuits with similar layer distributions and interventional effects) regardless of the random initialization.

In [ ]:
import random
import numpy as np
import torch

num_seeds = 5
seeds_to_test = [42, 100, 2023, 777, 9999]
k_heads = 20

seed_results = []
seed_heads_dict = {}

print(f"Running Gumbel search across {num_seeds} different seeds to test stability...")

# We will reuse the 'attacked_batch' and 'clean_batch' from the original Country-Capital task
for seed in seeds_to_test:
    print(f"\n--- Testing Seed {seed} ---")
    # Set seeds
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Initialize fresh mask engine
    mask_mod, n_l, n_h = setup_paper_gumbel_mask_engine(model, clear_cache=True)

    # Run search (using 300 epochs for faster iteration in stability testing)
    res = run_paper_gumbel_semantic_search(
        model, attacked_batch, mask_mod, n_l,
        lr=1.0, n_epochs=300, sparsity_lambda=1.0, show_progress=False, clear_cache=True
    )

    # Extract heads
    found_heads = get_bottom_k_heads(res["learned_masks"], k=k_heads)
    seed_heads_dict[seed] = set(found_heads)

    # Evaluate
    eval_res = evaluate_given_heads(
        model, attacked_batch, clean_batch, found_heads, method_name=f"Seed_{seed}"
    )
    seed_results.append(eval_res)

seed_df = compare_methods_table(seed_results)
display(seed_df)


Running Gumbel search across 5 different seeds to test stability...

--- Testing Seed 42 ---
Paper-style Gumbel mask engine initialized.
Shape: (32, 32)

--- Testing Seed 100 ---
Paper-style Gumbel mask engine initialized.
Shape: (32, 32)

--- Testing Seed 2023 ---
Paper-style Gumbel mask engine initialized.
Shape: (32, 32)

--- Testing Seed 777 ---
Paper-style Gumbel mask engine initialized.
Shape: (32, 32)

--- Testing Seed 9999 ---
Paper-style Gumbel mask engine initialized.
Shape: (32, 32)


,method,num_heads,head_density,attacked_base_margin,attacked_zero_margin,attacked_delta_margin,clean_base_margin,clean_zero_margin,clean_delta_margin,clean_base_gold,clean_zero_gold,clean_delta_gold
0,Seed_42,20,0.019531,0.374975,2.176047,1.801072,0.465683,1.525286,1.059603,0.93683,2.052093,1.115262
1,Seed_100,20,0.019531,0.374975,1.016467,0.641492,0.465683,1.004114,0.538431,0.93683,1.973177,1.036346
2,Seed_2023,20,0.019531,0.374975,2.833524,2.458549,0.465683,2.920406,2.454724,0.93683,4.778366,3.841535
3,Seed_777,20,0.019531,0.374975,0.760411,0.385436,0.465683,0.719522,0.253840,0.93683,1.061338,0.124508
4,Seed_9999,20,0.019531,0.374975,4.838070,4.463095,0.465683,5.647384,5.181701,0.93683,8.448248,7.511418


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

# 1. Pairwise Head Overlap (Intersection size / k_heads)
seed_list = list(seed_heads_dict.keys())
overlap_matrix = np.zeros((num_seeds, num_seeds))

for i in range(num_seeds):
    for j in range(num_seeds):
        set_i = seed_heads_dict[seed_list[i]]
        set_j = seed_heads_dict[seed_list[j]]
        intersection = len(set_i.intersection(set_j))
        overlap_matrix[i, j] = intersection / k_heads

fig_overlap = px.imshow(
    overlap_matrix,
    x=[f"Seed {s}" for s in seed_list],
    y=[f"Seed {s}" for s in seed_list],
    title="Pairwise Head Overlap (Intersection Ratio) Between Seeds",
    color_continuous_scale="Blues",
    text_auto=".2f"
)
fig_overlap.update_layout(width=600, height=500)
fig_overlap.show()

# 2. Layer Distribution Stability
# Count how many heads are in each layer for each seed
layer_counts_per_seed = {s: Counter([h[0] for h in heads]) for s, heads in seed_heads_dict.items()}

fig_layers = go.Figure()
for s in seed_list:
    counts = [layer_counts_per_seed[s].get(l, 0) for l in range(model.cfg.n_layers)]
    fig_layers.add_trace(go.Scatter(x=list(range(model.cfg.n_layers)), y=counts, mode='lines+markers', name=f'Seed {s}'))

fig_layers.update_layout(
    title="Layer Distribution of Discovered Heads Across Seeds",
    xaxis_title="Transformer Layer",
    yaxis_title="Number of Selected Heads",
    template="plotly_white",
    width=850,
    height=450
)
fig_layers.show()


In [ ]:
fig_perf = go.Figure(data=[
    go.Bar(name='Attacked (Base)', x=seed_df['method'], y=seed_df['attacked_base_margin']),
    go.Bar(name='Attacked (Ablated)', x=seed_df['method'], y=seed_df['attacked_zero_margin']),
    go.Bar(name='Clean (Base)', x=seed_df['method'], y=seed_df['clean_base_margin']),
    go.Bar(name='Clean (Ablated)', x=seed_df['method'], y=seed_df['clean_zero_margin'])
])

fig_perf.update_layout(
    title='Performance Stability: Ablation Impact Across Different Seeds',
    yaxis_title='Logit Margin (Gold - Distractor)',
    barmode='group',
    template='plotly_white',
    width=850,
    height=500
)
fig_perf.show()
